# Wireless Networking and Mobile Computing

### Prerequisites

This Jupyter Notebook has been tested with Visual Studio Code, running in a local Python environment.

**Installs**

In [ ]:
# --- Installs: ensure required packages are available ---
%pip install numpy soundfile torch wavmark resampy --quiet

### 1. Practical Over-The-Air Audio Communication

#### Setup Recommendations
This section is continously updated by usage reports.

**Computer (as Microphone)**
- [Audacity](https://www.audacityteam.org/) is a free software to record and edit audio and works perfectly.
- TBC

**iPhone (as Microphone)**
- GarageBand has been reported to not be optimal.
- VoiceMemos works well, but requires file type conversion. 
- Apps like [Audio Recorder](https://apps.apple.com/us/app/audio-recorder-wav-m4a/id1454488895) can directly export `.wav` and work well.

**Android (as Microphone)**
- TBC


#### 1.1 Setup

In [ ]:
# --- Imports: libraries and functions used in this notebook ---
from Audio.Enhanced_Wavmark import EnhancedWavMark, text_to_payloads, payloads_to_text, calculate_ber
import numpy as np
import sys
from scipy import signal as scipy_signal

# Initialize and show system info
enhanced_wavmark = EnhancedWavMark()
device_info = enhanced_wavmark.get_device_info()

print(f"System Information")
print(f"    - Python Version:   {sys.version.split()[0]}")
print(f"    - Device:           {device_info['device']} | CUDA Available: {'✅ YES' if device_info['cuda_available'] else '❌ NO'}")


# Helper Functions
def text_to_bits_with_robustness(text, robustness=1):
    basic_payloads = text_to_payloads(text)
    if robustness <= 1:
        return basic_payloads
    
    robust_payloads = []
    for _ in range(robustness):
        robust_payloads.extend(basic_payloads)
    return robust_payloads

def bits_to_text_with_majority_voting(bit_payloads, robustness=1):
    if robustness <= 1:
        return payloads_to_text(bit_payloads)
    
    original_length = len(bit_payloads) // robustness
    if original_length * robustness != len(bit_payloads):
        print(f"Expected {original_length * robustness} payloads, got {len(bit_payloads)}")
        return payloads_to_text(bit_payloads[:original_length])
    
    corrected_payloads = []
    payload_length = len(bit_payloads[0]) if bit_payloads else 16
    
    for i in range(original_length):
        copies = []
        for r in range(robustness):
            idx = r * original_length + i
            if idx < len(bit_payloads):
                copies.append(bit_payloads[idx])
        
        if copies:
            corrected_payload = np.zeros(payload_length, dtype=int)
            for bit_pos in range(payload_length):
                bit_votes = [copy[bit_pos] if bit_pos < len(copy) else 0 for copy in copies]
                corrected_payload[bit_pos] = 1 if sum(bit_votes) > len(bit_votes) // 2 else 0
            corrected_payloads.append(corrected_payload)
    
    return payloads_to_text(corrected_payloads)

def auto_align_audio(reference_file, recorded_file, output_file=None):
    print(f"Auto-aligning audio...")
    
    ref_signal, ref_sr = enhanced_wavmark.read_audio_mono_16k(reference_file)
    rec_signal, rec_sr = enhanced_wavmark.read_audio_mono_16k(recorded_file)

    print(f"   - Reference: {len(ref_signal)/16000:.1f}s ({len(ref_signal)} samples)")
    print(f"   - Recorded:  {len(rec_signal)/16000:.1f}s ({len(rec_signal)} samples)")

    # Use cross-correlation to find the best alignment: We correlate a chunk from the beginning of reference with the recorded signal
    chunk_size = min(16000 * 5, len(ref_signal)) 
    ref_chunk = ref_signal[:chunk_size]
    
    # Calculate cross-correlation
    correlation = scipy_signal.correlate(rec_signal, ref_chunk, mode='valid')
    
    # Find the peak correlation, which depicts the best alignment
    best_offset = np.argmax(correlation)
    max_correlation = correlation[best_offset]
    correlation_strength = max_correlation / (np.linalg.norm(ref_chunk) * np.linalg.norm(rec_signal[best_offset:best_offset+len(ref_chunk)]))
    
    print(f"   - Best offset: {best_offset} samples ({best_offset/16000:.2f}s)")
    print(f"   - Correlation: {correlation_strength:.3f}")
    
    # Determine how much of the reference we can extract
    available_length = len(rec_signal) - best_offset
    extract_length = min(len(ref_signal), available_length)
    
    # Extract the aligned portion
    aligned_audio = rec_signal[best_offset:best_offset + extract_length]
    
    print(f"   - Extracted: {len(aligned_audio)/16000:.1f}s ({len(aligned_audio)} samples)")
    print(f"   - Coverage: {len(aligned_audio)/len(ref_signal)*100:.1f}% of reference")
    
    # Save aligned audio
    if output_file:
        import soundfile as sf
        sf.write(output_file, aligned_audio, 16000)
        print(f"   - Saved to: {output_file}")
    
    return {
        'offset_samples': best_offset,
        'offset_seconds': best_offset / 16000,
        'correlation_strength': correlation_strength,
        'extracted_length': len(aligned_audio),
        'coverage_percentage': len(aligned_audio) / len(ref_signal) * 100,
        'aligned_audio': aligned_audio,
        'output_file': output_file,
        'alignment_quality': 'EXCELLENT' if correlation_strength > 0.75 else 
                           'GOOD' if correlation_strength > 0.5 else 
                           'FAIR' if correlation_strength > 0.2 else 'POOR'
    }

print("Setup successful")

#### 1.2 Inputs

Input and Output can be defined as you want. However, use only `.wav` files for playback and recording. If your recorder saves in another format (e.g., `.m4a`), convert it using a tool like [online-audio-converter.com](https://online-audio-converter.com/).

This demo works with a 16kHz audio and automatically converts any input to this rate.
   * We use this low-quality version for easier testing, as previous iterations of this exercise were plagued by compatibility issues.
   * 16kHz is useful as it will be fully captured by any high-sampling device (due to the [Shannon-Nyquist Sampling Theorem](https://en.wikipedia.org/wiki/Nyquist%E2%80%93Shannon_sampling_theorem)).

&nbsp;

The robustness adds redundancy to the message encoding, it does not reduce the BER. Make sure to always use the same robustness setting for encoding and decoding. Otherwise, this will prevent proper message reconstruction.



In [ ]:
# INPUT
INPUT_AUDIO_FILE = "CrabRave.wav"               # <-- Change this. It is the path to the input audio that should be encoded with the message
OUTPUT_AUDIO_FILE = "CrabRave_Encoded.wav"      # <-- Change this. It is the output file name you want to save the encoded audio as
ORIGINAL_MESSAGE = "This message was transmitted by audio over the air." # <-- Change this. It is the message to be transmitted
ROBUSTNESS_LEVEL = 1                            # <-- Change this. It is your desired robustness level (1 = no robustness, 2 = double each payload, ...).

In [ ]:
# Load audio and check raw capacity
signal, sr = enhanced_wavmark.read_audio_mono_16k(INPUT_AUDIO_FILE)
capacity = enhanced_wavmark.calculate_capacity(len(signal))

print(f"Audio Capacity ({INPUT_AUDIO_FILE})")
print(f"   - Duration:      {capacity['duration_seconds']:.1f}s")
print(f"   - Sections:      {capacity['num_sections']}")
print(f"   - Raw capacity:  {capacity['max_characters']} characters")

# Calculate effective capacity with robustness
max_chars_with_robustness = (capacity['num_sections'] // ROBUSTNESS_LEVEL) * 2 - 1
MESSAGE_TEXT = ORIGINAL_MESSAGE[:max_chars_with_robustness]

print(f"\nMessage Preparation (Robustness {ROBUSTNESS_LEVEL})")
print(f"   - Max chars with robustness:     {max_chars_with_robustness}")
print(f"   - Original: '        {ORIGINAL_MESSAGE}'")
print(f"   - Prepared: '        {MESSAGE_TEXT}' ({len(MESSAGE_TEXT)} chars)")

# Create robust payloads for encoding
robust_payloads = text_to_bits_with_robustness(MESSAGE_TEXT, ROBUSTNESS_LEVEL)
basic_payloads = text_to_payloads(MESSAGE_TEXT)

print(f"\nPayload Preparation")
print(f"   - Basic payloads:    {len(basic_payloads)}")
print(f"   - Robust payloads:   {len(robust_payloads)} (×{ROBUSTNESS_LEVEL})")
print(f"{'' if len(robust_payloads) <= capacity['num_sections'] else '   - Fits in audio: ❌ NO'}")

#### 1.3 Embedding Data in Audio

In [ ]:
# Encode robust payloads into audio
encoding_result = enhanced_wavmark.encode_multiple_payloads(
    INPUT_AUDIO_FILE, robust_payloads, OUTPUT_AUDIO_FILE
)

print(f"Encoding Complete")
print(f"   - Input:             {INPUT_AUDIO_FILE}")
print(f"   - Output:            {OUTPUT_AUDIO_FILE}")
print(f"   - Payloads encoded:  {len(robust_payloads)}")
status = "✅ SUCCESS" if encoding_result['failed_sections'] == 0 else f"❌ {encoding_result['failed_sections']} FAILED"
print(f"   - Status:            {status}")

#### 1.4 Extracting Data From Audio

In [ ]:
# INPUT
RECORDED_AUDIO_FILE = "CrabRave_Encoded.wav"            # <-- Change this. It is the path to your recorded file of the encoded audio played over the air.
ALIGNED_AUDIO_FILE = "CrabRave_Enc_Aligned.wav"         # <-- Change this. It is the file name of the aligned and trimmed recording you want to save.

The automatic signal alignment is not perfect, but limits manual trims and timing.

In [ ]:
# Audio Alignment and Decoding
alignment_result = auto_align_audio(INPUT_AUDIO_FILE, RECORDED_AUDIO_FILE, ALIGNED_AUDIO_FILE)

print(f"\nAudio Alignment Results")
print(f"   - Reference: {INPUT_AUDIO_FILE} (encoded)")
print(f"   - Recording: {RECORDED_AUDIO_FILE} (transmission)")
print(f"   - Quality: {alignment_result['alignment_quality']}")
print(f"   - Offset: {alignment_result['offset_seconds']:.2f}s")
print(f"   - Coverage: {alignment_result['coverage_percentage']:.1f}%")
print(f"   - Correlation: {alignment_result['correlation_strength']:.3f}")

In [ ]:
# Decode from aligned Audio
if alignment_result['alignment_quality'] in ['EXCELLENT', 'GOOD', 'FAIR']:
    print(f"Decoding from aligned recording...")
    decoding_result = enhanced_wavmark.decode_multiple_payloads(ALIGNED_AUDIO_FILE)
    
    print(f"\nDecoding Complete")
    print(f"   - Source: {ALIGNED_AUDIO_FILE}")
    print(f"   - Payloads found: {len(decoding_result['payloads'])}")
    print(f"   - Success rate: {decoding_result['success_rate']:.1%}")
    print(f"   - Avg confidence: {decoding_result['average_confidence']:.3f}")
else:
    print(f"\n❌ Alignment quality too poor for reliable decoding")
    print(f"   - Correlation {alignment_result['correlation_strength']:.3f} < 0.2")
    print(f"   - Try a different recording or check audio quality")
    decoding_result = None

#### 1.5 Message Decoding and BER
A BER of 100% is unrealistic, as that means every bit flipped. The worst BER in a binary channel is 50%, where no correlation remains between transmitted and received data.


In [ ]:
# Analyze Decoding Results
if decoding_result is None:
    print("❌ No decoding results available - alignment failed")
else:
    # Extract message with majority voting
    decoded_message = bits_to_text_with_majority_voting(decoding_result['payloads'], ROBUSTNESS_LEVEL)
    
    print(f"Message Extraction")
    print(f"   Original:    '{MESSAGE_TEXT}'")
    print(f"   Decoded:     '{decoded_message}'")
    print(f"   Match:       {'✅ PERFECT' if MESSAGE_TEXT == decoded_message else '❌ DIFFERENT'}")
    
    # BER Analysis
    ber_analysis = calculate_ber(MESSAGE_TEXT, decoded_message)
    
    print(f"\nError Analysis")
    print(f"   Character accuracy:  {ber_analysis['char_accuracy']:.1f}%")
    print(f"   Character errors:    {ber_analysis['char_errors']}/{ber_analysis['total_chars']}")
    print(f"   Bit Error Rate:      {ber_analysis['ber_percentage']:.2f}%")
    print(f"   Bit errors:          {ber_analysis['bit_errors']}/{ber_analysis['total_bits']}")
    
    # Bit Array Comparison with XOR visualization
    def text_to_bits(text):
        bits = []
        for char in text:
            char_bits = format(ord(char), '08b')
            bits.extend([int(b) for b in char_bits])
        return bits
    
    orig_bits = text_to_bits(MESSAGE_TEXT)
    dec_bits = text_to_bits(decoded_message)
    
    # Pad to same length
    max_len = max(len(orig_bits), len(dec_bits))
    while len(orig_bits) < max_len:
        orig_bits.append(0)
    while len(dec_bits) < max_len:
        dec_bits.append(0)
    
    # XOR comparison
    xor_bits = [orig_bits[i] ^ dec_bits[i] for i in range(max_len)]
    orig_str = ''.join(map(str, orig_bits))
    dec_str = ''.join(map(str, dec_bits))
    
    print(f"\nBit Array Comparison")
    print(f"    - Original: {orig_str}")
    print(f"    - Decoded:  {dec_str}")
    print(f"    - XOR Diff: ", end="")
    for bit in xor_bits:
        if bit == 1:
            print(f"\033[91m{bit}\033[0m", end="")
        else:
            print(f"\033[92m{bit}\033[0m", end="")
    print()